#### import libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
from ydata_profiling import ProfileReport
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import random


#### Read Data 

In [2]:
ratings=pd.read_csv(r"D:\ali\SIC\Project\Recoomend System\movies_metadata.csv\ratings_small.csv")
links=pd.read_csv(r"D:\ali\SIC\Project\Recoomend System\movies_metadata.csv\links_small.csv")
data_model=pd.read_csv(r"D:\ali\SIC\Project\Recoomend System\movies_metadata.csv\data_model.csv",keep_default_na=False)

In [3]:
data_model.isnull().sum()

id                      0
imdb_id                 0
genres                  0
original_language       0
overview                0
title                   0
production_countries    0
production_companies    0
runtime                 0
tagline                 0
dtype: int64

In [4]:
title_content=data_model['title']
text_content = data_model['overview'] + ' ' + data_model['tagline']
genre_content = data_model['genres']
data_model['content']=data_model['title']+' '+data_model['overview']+' '+data_model['genres']+' '+data_model['tagline']

In [5]:
Tf = TfidfVectorizer(stop_words='english')  
Tf.fit(data_model['content'].values.astype('U'))
print(f"num of unique words: {len(Tf.vocabulary_)}")

num of unique words: 81789


In [6]:
Tf_title=TfidfVectorizer(max_features=40000,stop_words='english')
vec_title=Tf_title.fit_transform(title_content.values.astype('U'))

In [7]:
Tf_text=TfidfVectorizer(max_features=40000,stop_words='english')
vec_text=Tf_title.fit_transform(title_content.values.astype('U'))

In [8]:
Tf_genre=TfidfVectorizer(max_features=40000,stop_words='english')
vec_genre=Tf_title.fit_transform(genre_content.values.astype('U'))

#### train the model

In [9]:
def recommend(movie_title, top_n=5):
    matches = data_model[data_model['title'] == movie_title]
    if matches.empty:
        return []
    index = matches.index[0] 
    sim_title = cosine_similarity(vec_title[index], vec_title).flatten()
    sim_text = cosine_similarity(vec_text[index], vec_text).flatten()
    sim_genre = cosine_similarity(vec_genre[index], vec_genre).flatten()
    sim=sim_title*0.15 + sim_text*.55 + sim_genre * .3  
    distance = sorted(list(enumerate(sim)), reverse=True, key=lambda x: x[1])
    return [data_model.iloc[i[0]]['title'] for i in distance[1:top_n+1]]

In [10]:
recommend('Toy Story',6)

['Toy Story 2',
 'Toy Story 3',
 'Toy Story of Terror!',
 'The Toy',
 'The Christmas Toy',
 'Toy Story That Time Forgot']

#### ُEvaluation


In [11]:
ratings_links = ratings.merge(
    links[['movieId', 'tmdbId']],
    on='movieId',
    how='inner'
)
ratings_model = ratings_links.merge(
    data_model[['id', 'title']],
    left_on='tmdbId',
    right_on='id',
    how='inner'
)

In [12]:
liked_movies = ratings_model[ratings_model['rating'] >= 4]

In [13]:

def precision_at_k_user(user_id, k=5):
    user_liked_list = liked_movies[liked_movies['userId'] == user_id]['title'].tolist()
    
    if len(user_liked_list) < 2:
        return None
    
    input_movie = user_liked_list[0]
    target_movies = set(user_liked_list[1:])
    
    recommendations = recommend(input_movie, k)
    if len(recommendations) == 0:
        return 0
    
    relevant = sum(movie in target_movies for movie in recommendations)
    return relevant / len(recommendations)

all_users = liked_movies['userId'].unique()
sample_users = random.sample(list(all_users), min(200, len(all_users)))

scores = []
for uid in sample_users:
    score = precision_at_k_user(uid, k=5)
    if score is not None:
        scores.append(score)

print(f"mean Precision@5: {sum(scores)/len(scores):.2%}")
print(f"number of user is evaluated: {len(scores)}")

mean Precision@5: 4.72%
number of user is evaluated: 199


In [14]:
def random_precision_at_k(user_id, k=5):
    user_liked_list = liked_movies[liked_movies['userId'] == user_id]['title'].tolist()
    
    if len(user_liked_list) < 2:
        return None
    
    target_movies = set(user_liked_list[1:])
    random_recs = random.sample(list(data_model['title']), k)
    
    relevant = sum(movie in target_movies for movie in random_recs)
    return relevant / k

random_scores = []
for uid in sample_users:
    score = random_precision_at_k(uid, k=5)
    if score is not None:
        random_scores.append(score)

print(f"Random Baseline Precision@5: {sum(random_scores)/len(random_scores):.2%}")

Random Baseline Precision@5: 0.50%


In [16]:
pickle.dump(data_model, open('data_model.pkl', 'wb'))
pickle.dump(vec_title, open('vec_title.pkl', 'wb'))
pickle.dump(vec_text, open('vec_text.pkl', 'wb'))
pickle.dump(vec_genre, open('vec_genre.pkl', 'wb'))